# P4 — ResNet50 + Transfer Learning + Ajuste fino

## Arquitectura principal

Se reutiliza ResNet50 preentrenada en ImageNet.

### Fase 1
Backbone congelado y entrenamiento de la cabeza.

### Fase 2
Fine tuning de `layer4` + cabeza con learning rate reducido.

In [1]:

from pathlib import Path
import sys
import yaml

def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()

    for candidate in [start, *start.parents]:
        if (candidate / "config.yaml").exists():
            return candidate

    raise FileNotFoundError(
        "No se encontró config.yaml. "
        "Ejecute Jupyter desde la carpeta del proyecto o notebooks/."
    )

ROOT = find_project_root()

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

cfg = yaml.safe_load(
    (ROOT / "config.yaml").read_text(encoding="utf-8")
)

print("Proyecto:", ROOT)


Proyecto: C:\Users\Nicolas Manco\Downloads\CONCRETE_DL


## 1. Configurar entorno

In [2]:
import torch

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

print("Dispositivo:", device)

Dispositivo: cpu


## 2. Cargar datos

In [3]:
from src.data import make_loaders

df, loaders = make_loaders(
    ROOT / cfg["data"]["supervised_split"],
    root=ROOT,
    input_size=cfg["model"]["input_size"],
    batch_size=cfg["training"]["batch_size"],
    num_workers=cfg["training"]["num_workers"]
)

## 3. Construir ResNet50

In [ ]:
from src.models import build_resnet50, freeze_backbone, unfreeze_layer4, count_parameters
from src.training import train_model
import pandas as pd

checkpoint = ROOT/"models/resnet50_finetuned.pt"
history_path = ROOT/"results/training/resnet50_history.csv"
history_path.parent.mkdir(parents=True, exist_ok=True)

if checkpoint.exists() and history_path.exists():
    model = build_resnet50(
        hidden=cfg["model"]["head_hidden"],
        dropout=cfg["model"]["dropout"],
        pretrained=False
    ).to(device)
    model.load_state_dict(torch.load(checkpoint, map_location=device))
    history = pd.read_csv(history_path)
    print("Se reutiliza resnet50_finetuned.pt y su historial reproducible.")
else:
    model = build_resnet50(
        hidden=cfg["model"]["head_hidden"],
        dropout=cfg["model"]["dropout"],
        pretrained=True
    ).to(device)

    freeze_backbone(model)
    model, hist_head = train_model(
        model, loaders["train"], loaders["val"],
        epochs=cfg["training"]["epochs_head"], lr=cfg["training"]["lr_head"],
        weight_decay=cfg["training"]["weight_decay"],
        patience=cfg["training"]["early_stopping_patience"],
        checkpoint=ROOT/"models/resnet50_head.pt", device=device)

    unfreeze_layer4(model)
    model, hist_ft = train_model(
        model, loaders["train"], loaders["val"],
        epochs=cfg["training"]["epochs_finetune"], lr=cfg["training"]["lr_finetune"],
        weight_decay=cfg["training"]["weight_decay"],
        patience=cfg["training"]["early_stopping_patience"],
        checkpoint=checkpoint, device=device)

    history = pd.concat([
        hist_head.assign(stage="head"),
        hist_ft.assign(stage="finetune")
    ], ignore_index=True)
    history.to_csv(history_path, index=False)

print('ResNet50 lista para evaluación.')
display(history.tail())


## 4. Contar parámetros totales

In [ ]:
total, trainable = count_parameters(model)
print("Parámetros totales:", total)
print("Parámetros entrenables en el estado cargado:", trainable)


## 5. Fase 1 — congelar backbone

### Historial de entrenamiento

El historial conserva las dos fases: entrenamiento del cabezal (`head`) y ajuste fino (`finetune`).


## 6. Entrenar cabeza

In [ ]:
print(history.groupby("stage").agg(epochs=("epoch","count"), best_val_loss=("val_loss","min")))


## 7. Fase 2 — Fine Tuning

### Ajuste fino

Cuando no existe un checkpoint, P4 entrena primero el cabezal y luego habilita `layer4/conv5_x` con una tasa de aprendizaje menor.


## 8. Entrenamiento fino

In [ ]:
best_by_stage = history.groupby("stage")["val_loss"].min().rename("best_val_loss").reset_index()
best_by_stage.to_csv(ROOT/"results/training/resnet50_best_by_stage.csv", index=False)
display(best_by_stage)


## 10. Curvas por fase

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
OUT=ROOT/"results/training"; OUT.mkdir(parents=True, exist_ok=True)

h=history.copy()
# eje global para representar transición real entre fases
h["global_epoch"] = np.arange(1, len(h)+1)
transition = int((h["stage"]=="head").sum()) + 0.5
best_idx = int(h["val_loss"].idxmin())
best_x = int(h.loc[best_idx,"global_epoch"]); best_y=float(h.loc[best_idx,"val_loss"])

fig, ax = plt.subplots(figsize=(8,5))
ax.plot(h["global_epoch"], h["train_loss"], label="Train BCE")
ax.plot(h["global_epoch"], h["val_loss"], label="Validation BCE")
ax.axvline(transition, linestyle="--", linewidth=1.2, label="Inicio fine-tuning")
ax.scatter([best_x],[best_y], s=45, zorder=5, label=f"Mejor val_loss={best_y:.4f}")
ax.set_title("ResNet50 — evolución de BCE"); ax.set_xlabel("Época global"); ax.set_ylabel("BCE")
ax.grid(alpha=.2); ax.legend(fontsize=8); fig.tight_layout()
fig.savefig(OUT/"resnet50_bce.png", dpi=220, bbox_inches="tight"); plt.show(); plt.close(fig)

fig, ax = plt.subplots(figsize=(8,5))
ax.plot(h["global_epoch"], h["val_accuracy"], label="ResNet50 validation")
ax.axvline(transition, linestyle="--", linewidth=1.2, label="Inicio fine-tuning")
ax.set_ylim(0,1.03); ax.set_title("ResNet50 — exactitud de validación")
ax.set_xlabel("Época global"); ax.set_ylabel("Accuracy"); ax.grid(alpha=.2); ax.legend(fontsize=8)
fig.tight_layout(); fig.savefig(OUT/"resnet50_val_accuracy.png", dpi=220, bbox_inches="tight"); plt.show(); plt.close(fig)
print('P4 exportó historial, resumen por fase y curvas de entrenamiento.')


In [ ]:

# Figura resumen reproducible para el artículo (generada por P4)
import shutil
base_hist_path=ROOT/'results/training/baseline_history.csv'
base_hist=pd.read_csv(base_hist_path) if base_hist_path.exists() else None
fig,axs=plt.subplots(1,2,figsize=(7.2,3.0))
axs[0].plot(h['global_epoch'],h['train_loss'],label='Train BCE')
axs[0].plot(h['global_epoch'],h['val_loss'],label='Validation BCE')
axs[0].axvline(transition,linestyle='--',linewidth=1.0,label='Inicio fine-tuning')
axs[0].scatter([best_x],[best_y],s=35,zorder=5,label='Mejor val_loss')
axs[0].set_xlabel('Época global'); axs[0].set_ylabel('BCE'); axs[0].grid(alpha=.2); axs[0].legend(fontsize=6)
axs[0].set_title('(a) ResNet50 — pérdida',fontsize=9)
if base_hist is not None:
    axs[1].plot(base_hist['epoch'],base_hist['val_accuracy'],label='CNN base')
axs[1].plot(h['global_epoch'],h['val_accuracy'],label='ResNet50')
axs[1].axvline(transition,linestyle='--',linewidth=1.0)
axs[1].set_ylim(0.45,1.02); axs[1].set_xlabel('Época'); axs[1].set_ylabel('Accuracy validación')
axs[1].grid(alpha=.2); axs[1].legend(fontsize=7); axs[1].set_title('(b) Exactitud de validación',fontsize=9)
fig.tight_layout()
for ext in ['png','pdf']:
    out=OUT/f'resnet50_training_summary.{ext}'; fig.savefig(out,dpi=240 if ext=='png' else None,bbox_inches='tight')
    shutil.copy2(out,ROOT/f'results/paper/fig3_entrenamiento.{ext}')
plt.close(fig)
print('Figura 3 del artículo actualizada desde P4.')
